<a href="https://colab.research.google.com/github/salalisiasa-debug/-2026/blob/main/lab_05(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа № 5. Временные ряды телеметрии и алгоритмы обнаружения аномалий

**Дисциплина:** «Методы обработки больших данных» · **Опорная лекция:** «Большие данные робототехнических и IoT-систем»

**Этап пайплайна:** `Store → Process → Learn` · **Среда:** Google Colab · `PySpark 4.2.0`, Spark Window API, NumPy, Pandas, Plotly · **Максимум:** 10 баллов

```text
Sense → Collect → Stream → Store → Process → Learn → Teach
                           ▲▲▲▲▲   ▲▲▲▲▲▲▲   ▲▲▲▲▲
```

**Оригинальные репозитории:** [InfluxCommunity/Notebooks](https://github.com/InfluxCommunity/Notebooks) (примеры [Anomaly_Detection](https://github.com/InfluxCommunity/Notebooks/tree/master/Anomaly_Detection)), [striim-labs/lstm-autoencoder-spark-kafka](https://github.com/striim-labs/lstm-autoencoder-spark-kafka).

**Цель работы:** научиться выделять аномальные состояния робототехнической телеметрии по временным признакам, используя Z-score и скользящий RMS; выполнить расчёт оконных признаков в Spark.

**Студенческая заготовка.** Полное описание — в `README_lab_05.md`.

### Как работать

Главное в этой работе — **не код, а понимание и выводы**: как найти «странные» моменты в работе робота, почему простое правило ошибается и как объяснить это школьнику. Код почти весь готов; дописать нужно **короткие строки** по подсказкам.

| Что видите в ячейке | Что сделать |
|---|---|
| `...` (три точки) | это **пустое место**: сотрите три точки и впишите код |
| `# ← впишите: Window.partitionBy("robot_id")` | справа после стрелки — **что именно** вписать вместо `...` (можно скопировать) |
| `Не заполнено: …` после запуска | в ячейке осталось `...`; в сообщении указано, какая строка |
| блок **«Что получено»** | дважды щёлкните по ячейке и допишите предложения-заготовки своими числами |

**Порядок:**

1. Выполните ячейку 0, затем укажите ФИО и группу в ячейке 1.
2. Выполняйте ячейки сверху вниз (**Shift + Enter**). Если ячейка остановилась с сообщением `Не заполнено`, впишите код по подсказке и запустите её снова.
3. После каждой ячейки прочитайте блок **«Как читать результат»** и заполните **«Что получено»** своими числами — это и есть основная часть отчёта.
4. Заполните `TEACH CARD` и запустите самопроверку (ячейка 9).
5. Сдайте ноутбук с сохранёнными графиками и выводами.

**Если появилась ошибка** `NameError: name '…' is not defined` — не запущена одна из **предыдущих** ячеек: выберите **Runtime → Run before** и повторите.

| Где дописать код | Что вписать |
|---|---|
| ячейка 1 | ФИО и группа |
| ячейка 3, TODO 3.1–3.4 | окно робота, среднее, σ и z (4 строки) |
| ячейка 4, TODO 4.1–4.2 | скользящее окно и RMS (2 строки) |
| ячейка 5, TODO 5 | правило аномалии (1 строка) |
| ячейка 7, TODO 7.1–7.2 | подсчёт TP / FP / FN и списки порогов (5 строк) |

### Критерии оценки (10 баллов)

| Баллы | Что оценивается | Где в ноутбуке |
|---:|---|---|
| 2 | корректно сформирован временной ряд с контролируемыми аномалиями | ячейка 2 |
| 2 | корректно рассчитан Z-score по каждому роботу | ячейка 3 |
| 2 | корректно рассчитан rolling RMS средствами Spark Window API | ячейка 4 |
| 2 | выполнен анализ чувствительности порогов и построены информативные графики | ячейки 5–8 |
| 2 | корректно заполненная форма `TEACH CARD` (педагогическая адаптация инженерной задачи) | часть «TEACH CARD» |

---

# Теоретический минимум

## Z-score

$$z_i = \frac{x_i - \mu}{\sigma}.$$

Типовое правило $|z_i| \ge 3$ выделяет сильные отклонения относительно базового распределения. Важно, **по каким данным** считаются $\mu$ и $\sigma$:

| Способ калибровки | $\mu$, $\sigma$ считаются по… | Слабое место |
|---|---|---|
| глобальный (как в задании) | всему ряду робота, вместе с аномалией | аномалия сама раздувает $\sigma$ («маскирование»); тренд тоже входит в $\sigma$ |
| по нормальному участку | первым минутам работы | не учитывает дрейф: через полчаса «нормальная» температура выходит за порог |
| скользящий | предыдущим $W$ отсчётам | длинная аномалия попадает в своё же окно и «гасит» себя |

## Скользящий RMS

$$\mathrm{RMS}_t = \sqrt{\frac{1}{N}\sum_{i=t-N+1}^{t} x_i^2}.$$

Для вибрационного сигнала RMS на окне длины $N$ показывает изменение энергии сигнала во времени. Окно «назад» (как в задании) работает **онлайн**, но запаздывает: RMS растёт постепенно, пока окно заполняется аномальными отсчётами, и остаётся высоким ещё почти $N$ отсчётов после конца аномалии.

## Окна в Spark

| Окно | Код | Назначение |
|---|---|---|
| вся партиция | `Window.partitionBy("robot_id")` | $\mu$ и $\sigma$ робота |
| скользящее по строкам | `.orderBy(ts).rowsBetween(-19, 0)` | 20 последних отсчётов (верно при равномерной частоте) |
| скользящее по времени | `.orderBy(ts).rangeBetween(-19, 0)` | последние 20 секунд, даже если есть пропуски |

`partitionBy` гарантирует, что статистики одного робота не смешиваются с другим.

## Качество обнаружения

Пороги в production калибруются на нормальном режиме и проверяются по **precision** и **recall**, а не задаются произвольно:

$$\text{precision} = \frac{TP}{TP+FP}, \qquad \text{recall} = \frac{TP}{TP+FN}, \qquad F_1 = \frac{2PR}{P+R}.$$

Метрики можно считать **по отсчётам** (каждая секунда) и **по событиям** (связные интервалы тревоги). Скользящее окно почти всегда «портит» поточечные метрики своим хвостом, хотя само событие обнаружено.

---
# Подготовка среды

## Ячейка 0. Установка библиотек и Java

**Что делает ячейка.** Ставит `pyspark==4.2.0` и `plotly`, если их нет, и проверяет версию Java. **Spark 4.x работает только на Java 17 или 21**. Если в среде более старая Java, ячейка ставит OpenJDK 17 и задаёт `JAVA_HOME`. Без этого `SparkSession` не запустится, и вместо понятной ошибки появится длинный Java-стек.

**Задание.** Ячейка готова — просто запустите (первый запуск 1–2 минуты) и убедитесь, что Java не ниже 17.

In [ ]:
import importlib.util, os, re, shutil, subprocess, sys
from importlib.metadata import version, PackageNotFoundError

def installed(dist):
    try:
        return version(dist)
    except PackageNotFoundError:
        return None

# 1) Python-пакеты задания
need = []
if installed("pyspark") != "4.2.0":
    need.append("pyspark==4.2.0")
for pkg in ["plotly", "pandas", "numpy"]:
    if importlib.util.find_spec(pkg) is None:
        need.append(pkg)
if need:
    print("Устанавливаю:", " ".join(need))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])

# 2) Spark 4.x работает только на Java 17 или 21. Если в среде старая Java — ставим OpenJDK 17.
def java_major():
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        m = re.search(r'version "(\d+)', out)
        return int(m.group(1)) if m else None
    except FileNotFoundError:
        return None

if (java_major() or 0) < 17:
    print("Java", java_major(), "не подходит для Spark 4 — устанавливаю OpenJDK 17 (Colab)")
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

for dist in ["pyspark", "plotly", "pandas", "numpy"]:
    print(f"{dist:<8} {installed(dist)}")
print("Java    ", java_major())

Устанавливаю: pyspark==4.2.0
pyspark  4.2.0
plotly   5.24.1
pandas   2.2.3
numpy    2.1.3
Java     21


**Что получено.** Допишите:

> Версия pyspark — 4.2.0, версия Java — 21




(Spark 4 требует Java не ниже 17: условие выполнено).

## Ячейка 1. Сессия Spark

**Что делает ячейка.** Создаёт локальную `SparkSession` с параметрами задания (`local[*]`, 4 shuffle-раздела) и задаёт `PYSPARK_PYTHON`. Тогда драйвер и рабочие процессы Spark используют одну и ту же версию Python. При разных версиях Spark 4 останавливается с ошибкой `PYTHON_VERSION_MISMATCH`.

**Задание.** Впишите ФИО и группу в кавычках вместо примера и запустите ячейку. Остальной код готов.

In [ ]:
import os, sys
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # одна версия Python у драйвера и воркеров

# =========== ЗАПОЛНИТЕ ===========
STUDENT_NAME  = "Салимуллина Алия Дамировна"   # TODO: ← впишите свои ФИО (в кавычках)
STUDENT_GROUP = "ИНДОР-221"                  # TODO: ← впишите свою группу (в кавычках)
# =================================

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pyspark.sql import SparkSession, functions as F, Window

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab05_TimeSeriesAnomalyDetection")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")   # без полос прогресса в выводе ноутбука
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| ядер:", spark.sparkContext.defaultParallelism)


def check_filled(where, values):
    """Проверяет, что вместо «...» вписан код; если нет — называет незаполненную строку."""
    empty = [name for name, v in values.items() if v is ...]
    if empty:
        raise ValueError(f"Не заполнено ({where}): " + ", ".join(empty)
                         + ". Замените «...» на код из подсказки «← впишите:» в этой строке и запустите ячейку снова.")

Spark 4.2.0 | ядер: 2


**Что получено.** Допишите:

> Сессия Spark версии 4.2.0 запущена в режиме `local[*]` на 2 ядрах — «кластер» здесь моделируется на одном компьютере.

---
# Шаг 1. Временной ряд с контролируемыми аномалиями (2 балла)

## Ячейка 2. Генерация телеметрии

**Что делает ячейка.** Генерирует телеметрию **кодом задания без изменения параметров**:
* 3 робота × 1800 с при 1 Гц;
* температура: $44 + i + 0{,}002\,t + \mathcal N(0;\,0{,}25)$;
* вибрация: $0{,}20 + 0{,}03\sin(t/15) + \mathcal N(0;\,0{,}015)$.

Аномалии:
* перегрев +10 °C на 850–869 с (20 отсчётов);
* рост вибрации +0,75 на 1200–1224 с (25 отсчётов).

Добавлены только два столбца **разметки истины** `true_temp` и `true_vib`. В правиле обнаружения они не участвуют, но без них нельзя посчитать precision и recall (шаг 7).

**Задание.** Ячейка готова — просто запустите и рассмотрите график. Параметры генерации **не меняйте**.

In [ ]:
# 1. Синтетическая телеметрия нескольких роботов (параметры задания без изменений).
rng = np.random.default_rng(42)
robots = ["robot-0", "robot-1", "robot-2"]
n_per_robot = 1800  # 30 минут при 1 Гц

frames = []
start = pd.Timestamp("2026-08-31T09:00:00Z")

for ridx, robot in enumerate(robots):
    t = np.arange(n_per_robot)
    temperature = 44 + ridx + 0.002 * t + rng.normal(0, 0.25, n_per_robot)
    vibration = 0.20 + 0.03 * np.sin(t / 15) + rng.normal(0, 0.015, n_per_robot)

    # Вставка аномалий.
    temperature[850:870] += 10
    vibration[1200:1225] += 0.75

    # Разметка истины: какие отсчёты мы испортили (нужна для оценки качества, в Spark-правиле не используется)
    true_temp = np.zeros(n_per_robot, dtype=bool); true_temp[850:870] = True
    true_vib = np.zeros(n_per_robot, dtype=bool); true_vib[1200:1225] = True

    frames.append(pd.DataFrame({
        "robot_id": robot,
        "timestamp": start + pd.to_timedelta(t, unit="s"),
        "temperature": temperature,
        "vibration": vibration,
        "true_temp": true_temp,
        "true_vib": true_vib,
    }))

pdf = pd.concat(frames, ignore_index=True)
print(f"Строк: {len(pdf)} ({len(robots)} робота × {n_per_robot} с); "
      f"аномальных по температуре: {pdf.true_temp.sum()}, по вибрации: {pdf.true_vib.sum()}")
print(pdf.groupby("robot_id")[["temperature", "vibration"]].agg(["mean", "std", "min", "max"]).round(3))

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Температура, °C", "Вибрация, усл. ед."))
colors = {"robot-0": "#2a78d6", "robot-1": "#eb6834", "robot-2": "#1baf7a"}
for robot, g in pdf.groupby("robot_id"):
    fig.add_trace(go.Scatter(x=g.timestamp, y=g.temperature, name=robot, line=dict(width=1, color=colors[robot])), 1, 1)
    fig.add_trace(go.Scatter(x=g.timestamp, y=g.vibration, name=robot, showlegend=False,
                             line=dict(width=1, color=colors[robot])), 2, 1)
for row, (a, b) in ((1, (850, 870)), (2, (1200, 1225))):
    fig.add_vrect(x0=start + pd.Timedelta(seconds=a), x1=start + pd.Timedelta(seconds=b),
                  fillcolor="orange", opacity=0.25, line_width=0, row=row, col=1)
fig.update_layout(height=520, title="Исходная телеметрия: оранжевым — вставленные аномалии (истина)",
                  template="simple_white", legend=dict(orientation="h", y=1.08))
fig.show()

Строк: 5400 (3 робота × 1800 с); аномальных по температуре: 60, по вибрации: 75
         temperature                        vibration                     
                mean    std     min     max      mean    std    min    max
robot_id                                                                  
robot-0       45.900  1.484  43.520  55.938     0.211  0.089  0.136  0.964
robot-1       46.906  1.497  44.565  57.343     0.211  0.089  0.121  0.965
robot-2       47.904  1.503  45.501  58.096     0.210  0.089  0.130  0.969


**Как читать результат.**

* Верхний график — температура трёх роботов, нижний — вибрация. Оранжевые полосы — места, где мы **сами** вставили аномалии; это «правильные ответы», с которыми потом сравним алгоритм.
* Перегрев — короткий «прямоугольный» скачок на +10 °C; вибрация — короткий всплеск на +0,75.
* Температура медленно растёт из-за слагаемого $0{,}002\,t$: за 30 минут мотор нагревается примерно на 3,6 °C — это нормальный дрейф, а не неисправность.

**Что получено.** Допишите своими числами:

> Всего 5400 строк (3 робота × 1800 с). Аномальных отсчётов по температуре — 60,, по вибрации — 75.
> Перегрев виден на 850-й секунде, всплеск вибрации — на 1200-й. Температура медленно растёт, потому что в модели предусмотрен постепенный нагрев двигателя на 0,002 °C в секунду (примерно 3,6 °C за 30 минут).

---
# Шаг 2–3. Z-score температуры по каждому роботу (2 балла)

## Ячейка 3. Среднее, σ и Z-score в окне робота

**Что делает ячейка.** Считает $\mu$, $\sigma$ и $z$ **кодом задания** (`Window.partitionBy("robot_id")`, `stddev_pop`) и сверяет результат с pandas. Затем показывает, почему $\sigma$ получилась такой. В конце сравнивает глобальную калибровку с двумя другими: по первым 10 минутам и по скользящему окну 300 с.

**Задание.** Допишите **4 строки** (TODO 3.1–3.4). Подсказка к каждой — справа от неё. Сравнение калибровок ниже уже готово.

* **Окно** `Window.partitionBy("robot_id")` — «считай отдельно для каждого робота»: среднее робота-0 не смешивается с роботом-1.
* `F.avg(...)` — среднее, `F.stddev_pop(...)` — стандартное отклонение σ (насколько значения обычно «разбросаны» вокруг среднего).
* **z** показывает, на сколько σ значение отличается от среднего: |z| ≥ 3 — очень необычное значение.

In [ ]:
sdf = spark.createDataFrame(pdf)

# 2. Z-score температуры внутри каждого робота (код задания).
w_robot = Window.partitionBy("robot_id")      # TODO 3.1 ← впишите: Window.partitionBy("robot_id")
temp_mean = F.avg("temperature").over(w_robot)    # TODO 3.2 ← впишите: F.avg("temperature").over(w_robot)
temp_std = F.stddev_pop("temperature").over(w_robot)     # TODO 3.3 ← впишите: F.stddev_pop("temperature").over(w_robot)
temp_z = (F.col("temperature") - F.col("temp_mean")) / F.col("temp_std")       # TODO 3.4 ← впишите: (F.col("temperature") - F.col("temp_mean")) / F.col("temp_std")
check_filled("ячейка 3, TODO 3.1–3.4", {"w_robot": w_robot, "temp_mean": temp_mean, "temp_std": temp_std, "temp_z": temp_z})

sdf = (
    sdf
    .withColumn("temp_mean", temp_mean)    # готово: новый столбец — среднее робота
    .withColumn("temp_std", temp_std)      # готово: новый столбец — σ робота
    .withColumn("temp_z", temp_z)          # готово: новый столбец — z каждого отсчёта
)

# Контроль: тот же расчёт в pandas (ddof=0 соответствует stddev_pop)
check = pdf.groupby("robot_id")["temperature"].agg(mean="mean", std=lambda s: s.std(ddof=0))
stats = sdf.groupBy("robot_id").agg(F.first("temp_mean").alias("temp_mean"),
                                    F.first("temp_std").alias("temp_std"),
                                    F.max("temp_z").alias("z_max"),
                                    F.min("temp_z").alias("z_min")).orderBy("robot_id").toPandas().set_index("robot_id")
stats["pandas_std"] = check["std"]
print(stats.round(4))

# Почему σ = 1,9, а не 0,25: вклад тренда и самой аномалии
trend_sd = 0.002 * n_per_robot / np.sqrt(12)
anom_share = 20 / n_per_robot
anom_sd = 10 * np.sqrt(anom_share * (1 - anom_share))
print(f"\nσ шума 0,25; σ линейного тренда {trend_sd:.3f}; σ, вносимая аномалией {anom_sd:.3f}; "
      f"суммарно ≈ {np.sqrt(0.25**2 + trend_sd**2 + anom_sd**2):.3f}")

# Калибровка на нормальном режиме: μ и σ по первым 10 минутам (до аномалий), как требует теория
w_base = Window.partitionBy("robot_id")
base = F.col("timestamp") < F.lit(start + pd.Timedelta(minutes=10))
sdf = (sdf
       .withColumn("base_mean", F.avg(F.when(base, F.col("temperature"))).over(w_base))
       .withColumn("base_std", F.stddev_pop(F.when(base, F.col("temperature"))).over(w_base))
       .withColumn("temp_z_base", (F.col("temperature") - F.col("base_mean")) / F.col("base_std")))
# Локальная калибровка: μ и σ по предыдущим 300 с (5 минут), текущая точка в окно не входит
w_prev = Window.partitionBy("robot_id").orderBy(F.col("timestamp").cast("long")).rowsBetween(-300, -1)
sdf = (sdf
       .withColumn("roll_mean", F.avg("temperature").over(w_prev))
       .withColumn("roll_std", F.stddev_pop("temperature").over(w_prev))
       .withColumn("roll_n", F.count("temperature").over(w_prev))
       # Spark 4 работает в режиме ANSI: деление на σ = 0 даёт ошибку, а не NULL — нужен try_divide
       .withColumn("temp_z_roll", F.when(F.col("roll_n") >= 30,
                                         F.try_divide(F.col("temperature") - F.col("roll_mean"), F.col("roll_std")))))
cmp_z = (sdf.groupBy("robot_id")
         .agg(F.first("base_std").alias("σ первых 10 мин"),
              F.max(F.when(~F.col("true_temp"), F.abs("temp_z_base"))).alias("макс.|z| вне аномалии: калибр. 10 мин"),
              F.max(F.when(~F.col("true_temp"), F.abs("temp_z_roll"))).alias("макс.|z| вне аномалии: скользящий 300 с"),
              F.max(F.when(~F.col("true_temp"), F.abs("temp_z"))).alias("макс.|z| вне аномалии: задание"))
         .orderBy("robot_id").toPandas())
print(cmp_z.round(2).to_string(index=False))

          temp_mean  temp_std   z_max   z_min  pandas_std
robot_id                                                 
robot-0     45.8997    1.4837  6.7657 -1.6038      1.4837
robot-1     46.9062    1.4969  6.9720 -1.5641      1.4969
robot-2     47.9041    1.5025  6.7838 -1.5997      1.5025

σ шума 0,25; σ линейного тренда 1.039; σ, вносимая аномалией 1.048; суммарно ≈ 1.497
robot_id  σ первых 10 мин  макс.|z| вне аномалии: калибр. 10 мин  макс.|z| вне аномалии: скользящий 300 с  макс.|z| вне аномалии: задание
 robot-0             0.42                                   8.67                                     4.13                            1.60
 robot-1             0.42                                   8.34                                     3.86                            1.56
 robot-2             0.42                                   8.21                                     3.78                            1.60


**Как читать результат.**

* Первая таблица: среднее и σ каждого робота; столбец `pandas_std` — та же σ, посчитанная другим способом (должна совпасть — это проверка).
* Строка «σ шума… σ тренда… σ аномалии» объясняет, **из чего** сложилась σ: случайный шум, медленный нагрев и сам перегрев вместе «раздувают» разброс.
* Последняя таблица сравнивает три способа выбрать «норму» для z. Чем меньше |z| на **нормальных** отсчётах, тем меньше ложных тревог.

**Что получено.** Допишите своими числами:

> У роботов σ температуры = 1.4837 / 1.4969 /1.5025 °C, максимальный z = 6.7657 / 6.9720  / 6.7838 . σ совпала с pandas: да.
> σ намного больше шума 0,25, потому что в неё вошли тренд (≈ 1,039) и сама аномалия (≈ 1.048).
> Наименьший |z| вне аномалии дала калибровка 1.56, потому что она учитывает начальное нормальное состояние робота без перегрева.
> Калибровка по первым 10 минутам не работает, потому что из-за постепенного нагрева температура со временем отклоняется от первоначальной нормы, вызывая ложные срабатывания.

---
# Шаг 4. Скользящий RMS вибрации (2 балла)

## Ячейка 4. Окно на 20 измерений

**Что делает ячейка.** Считает `vibration_rms` **кодом задания** (`rowsBetween(-19, 0)`: текущий отсчёт и 19 предыдущих). Дополнительно считает число отсчётов в окне и RMS по центрированному окну для сравнения. Теоретически оценивает, когда RMS пересечёт порог 0,45.

**Задание.** Допишите **2 строки** (TODO 4.1–4.2).

* **Скользящее окно** — «рамка» из 20 последних измерений, которая сдвигается на один шаг с каждой секундой: `orderBy(...)` задаёт порядок по времени, `rowsBetween(-19, 0)` — 19 предыдущих строк и текущая.
* **RMS** — корень из среднего квадратов: показывает «силу» вибрации в рамке. `F.pow(x, 2)` — квадрат, `F.avg` — среднее, `F.sqrt` — корень.

In [ ]:
# 3. Скользящий RMS вибрации за последние 20 измерений (код задания).
w_roll = Window.partitionBy("robot_id").orderBy(F.col("timestamp").cast("long")).rowsBetween(-19, 0)       # TODO 4.1 ← впишите: Window.partitionBy("robot_id").orderBy(F.col("timestamp").cast("long")).rowsBetween(-19, 0)
vibration_rms = F.sqrt(F.avg(F.pow("vibration", 2)).over(w_roll))  # TODO 4.2 ← впишите: F.sqrt(F.avg(F.pow("vibration", 2)).over(w_roll))
check_filled("ячейка 4, TODO 4.1–4.2", {"w_roll": w_roll, "vibration_rms": vibration_rms})

sdf = sdf.withColumn("vibration_rms", vibration_rms)    # готово: новый столбец — RMS вибрации
# Сколько отсчётов реально попало в окно (первые 19 строк каждого робота — неполные окна)
sdf = sdf.withColumn("rms_window_n", F.count("vibration").over(w_roll))
# Для сравнения: центрированное окно (10 с до и 9 с после) — возможно только в офлайн-анализе
w_center = Window.partitionBy("robot_id").orderBy(F.col("timestamp").cast("long")).rowsBetween(-10, 9)
sdf = sdf.withColumn("vibration_rms_c", F.sqrt(F.avg(F.pow("vibration", 2)).over(w_center))).cache()

rms_stats = (sdf.groupBy("robot_id")
             .agg(F.expr("percentile_approx(vibration_rms, 0.5)").alias("RMS медиана"),
                  F.max(F.when(~F.col("true_vib"), F.col("vibration_rms"))).alias("RMS макс. вне аномалии"),
                  F.max("vibration_rms").alias("RMS макс."),
                  F.sum((F.col("rms_window_n") < 20).cast("int")).alias("неполных окон"))
             .orderBy("robot_id").toPandas())
print(rms_stats.round(3).to_string(index=False))

# Теория: при скольких аномальных отсчётах в окне RMS превысит 0,45
base_ms, anom_ms = 0.20**2 + 0.03**2 / 2 + 0.015**2, (0.20 + 0.75)**2 + 0.03**2 / 2 + 0.015**2
k_min = int(np.ceil(20 * (0.45**2 - base_ms) / (anom_ms - base_ms)))
print(f"\nRMS ≥ 0,45, когда в окне ≥ {k_min} аномальных отсчётов из 20: первое срабатывание ≈ через {k_min - 1}–{k_min} с "
      f"после начала, «хвост» ≈ {20 - k_min} с после конца")

robot_id  RMS медиана  RMS макс. вне аномалии  RMS макс.  неполных окон
 robot-0        0.203                   0.910      0.932             19
 robot-1        0.203                   0.909      0.930             19
 robot-2        0.202                   0.908      0.930             19

RMS ≥ 0,45, когда в окне ≥ 4 аномальных отсчётов из 20: первое срабатывание ≈ через 3–4 с после начала, «хвост» ≈ 16 с после конца


**Как читать результат.**

* «RMS медиана» — типичная сила вибрации в нормальной работе; «RMS макс.» — пик во время аномалии.
* «Неполных окон» — первые 19 секунд у каждого робота, когда в рамке ещё меньше 20 измерений.
* Последняя строка: рамка должна наполниться аномальными значениями, прежде чем RMS превысит порог 0,45 — отсюда **задержка** тревоги и **«хвост»** после конца аномалии.

**Что получено.** Допишите своими числами:

> Медиана RMS ≈0,203,, максимум ≈ 0,932; неполных окон у каждого робота — 19.
> Медиана RMS ≈ 0,20, а не 0, потому что даже при нормальной работе робот постоянно испытывает небольшие вибрации.
> RMS превышает 0,45, когда в рамке ≥ 4 аномальных отсчётов из 20: тревога запаздывает примерно на 3–4 с, а «хвост» длится ≈ 16 с.

---
# Шаг 5. Правило аномалии и таблица событий

## Ячейка 5. Логическое правило

**Что делает ячейка.** Применяет правило задания `|temp_z| ≥ 3 OR vibration_rms ≥ 0,45`, получает таблицу событий и раскладывает срабатывания по причинам. Затем сравнивает их с истиной и измеряет задержку и «хвост» правила по вибрации.

**Задание.** Допишите **одну строку** (TODO 5) — правило «тревога, если перегрев **или** сильная вибрация». В Spark `|` означает «или», `&` — «и», `F.abs(...)` — модуль числа; каждое условие берите в скобки.

**Словарик к таблице:** **TP** — тревога была и аномалия была (верно); **FP** — ложная тревога; **FN** — пропущенная аномалия.

In [ ]:
# 4. Аномалия по двум независимым признакам (код задания).
rule = (F.abs("temp_z") >= 3.0) | (F.col("vibration_rms") >= 0.45)         # TODO 5 ← впишите: (F.abs("temp_z") >= 3.0) | (F.col("vibration_rms") >= 0.45)
check_filled("ячейка 5, TODO 5", {"rule": rule})

sdf = sdf.withColumn("is_anomaly", rule)    # готово: новый столбец True/False — тревога или нет

anomalies = (
    sdf.filter("is_anomaly")
    .select("robot_id", "timestamp", "temperature", "temp_z", "vibration", "vibration_rms")
    .orderBy("robot_id", "timestamp")
).cache()

print("Detected anomalies:", anomalies.count())
anomalies.show(5, truncate=False)

# Разбор по причинам и сравнение с истиной
by_cause = (sdf.groupBy("robot_id")
            .agg(F.sum((F.abs("temp_z") >= 3.0).cast("int")).alias("по температуре"),
                 F.sum((F.col("vibration_rms") >= 0.45).cast("int")).alias("по вибрации"),
                 F.sum(F.col("is_anomaly").cast("int")).alias("всего"),
                 F.sum((F.col("is_anomaly") & (F.col("true_temp") | F.col("true_vib"))).cast("int")).alias("TP"),
                 F.sum((F.col("is_anomaly") & ~(F.col("true_temp") | F.col("true_vib"))).cast("int")).alias("FP"),
                 F.sum((~F.col("is_anomaly") & (F.col("true_temp") | F.col("true_vib"))).cast("int")).alias("FN"))
            .orderBy("robot_id").toPandas())
print(by_cause.to_string(index=False))

# Интервалы срабатываний правила по вибрации относительно истинного интервала 1200–1224 с
vib_hits = (sdf.filter(F.col("vibration_rms") >= 0.45)
            .withColumn("sec", (F.col("timestamp").cast("long") - int(start.timestamp())))
            .groupBy("robot_id").agg(F.min("sec").alias("первое, с"), F.max("sec").alias("последнее, с"))
            .orderBy("robot_id").toPandas())
vib_hits["задержка, с"] = vib_hits["первое, с"] - 1200
vib_hits["хвост, с"] = vib_hits["последнее, с"] - 1224
print(vib_hits.to_string(index=False))

Detected anomalies: 171
+--------+-------------------+------------------+-----------------+-------------------+-------------------+
|robot_id|timestamp          |temperature       |temp_z           |vibration          |vibration_rms      |
+--------+-------------------+------------------+-----------------+-------------------+-------------------+
|robot-0 |2026-08-31 09:14:10|55.609541865728914|6.544490923064819|0.18767095667811073|0.18977264294835694|
|robot-0 |2026-08-31 09:14:11|55.57956245427469 |6.524284520686965|0.17704131645998739|0.19061785539166629|
|robot-0 |2026-08-31 09:14:12|55.93115033997303 |6.761258028337304|0.22675968873127003|0.19343030886348225|
|robot-0 |2026-08-31 09:14:13|55.71377139694279 |6.614742597303944|0.196977855989439  |0.1945024740884341 |
|robot-0 |2026-08-31 09:14:14|55.77764397862795 |6.657793311905647|0.21094687949927654|0.19544568488590996|
+--------+-------------------+------------------+-----------------+-------------------+-------------------+
only

**Как читать результат.**

* `Detected anomalies` — сколько секунд правило отметило как тревожные (у всех трёх роботов вместе).
* Таблица по роботам раскладывает тревоги **по причинам** и сравнивает с «правильными ответами»: TP — верные тревоги, FP — ложные, FN — пропущенные.
* Последняя таблица: когда сработала тревога по вибрации относительно настоящей аномалии (1200–1224 с) — это и есть задержка и «хвост» из ячейки 4.

**Что получено.** Допишите своими числами:

> Всего обнаружено 171 аномальных отсчётов: у каждого робота 20 по температуре и 37 по вибрации.
> У одного робота TP = 41, FP = 16, FN = 4.
> Тревога по вибрации началась на 1204 с (задержка 4 с) и закончилась на 1240 с («хвост» 16 с). Поэтому FN появляются в начале аномалии, а FP —после окончания аномалии, поскольку скользящее окно RMS продолжает учитывать предыдущие повышенные значения вибрации

---
# Шаг 6. Графики

## Ячейка 6. RMS, аномальные точки, крупный план

**Что делает ячейка.** Строит два графика задания и третий — крупный план одного робота.
* **Первый** (RMS вибрации) дополнен порогом и интервалом истины.
* **Второй** (аномальные точки на температуре) раскрашен **по сработавшему правилу**: в исходном варианте вибрационные тревоги стояли на графике температуры без объяснения, почему они там оказались.
* **Третий** — крупный план одного робота.

**Задание.** Ячейка готова — просто запустите и разберите три графика по подсказкам ниже.

In [ ]:
# 5. Визуализация (графики задания + разметка истины).
plot_df = (
    sdf.select("robot_id", "timestamp", "temperature", "temp_z", "vibration_rms", "is_anomaly", "true_temp", "true_vib")
    .toPandas()
)

fig = px.line(plot_df, x="timestamp", y="vibration_rms", color="robot_id",
              color_discrete_map=colors, title="Rolling RMS вибрации (окно 20 отсчётов)")
fig.add_hline(y=0.45, line_dash="dash", line_color="#e34948", line_width=2, opacity=1, annotation_text="порог 0,45")
fig.add_vrect(x0=start + pd.Timedelta(seconds=1200), x1=start + pd.Timedelta(seconds=1225),
              fillcolor="orange", opacity=0.25, line_width=0, annotation_text="истина")
fig.update_layout(template="simple_white", height=380, xaxis_title="время", yaxis_title="RMS вибрации, усл. ед.")
fig.show()

anom_pdf = plot_df[plot_df["is_anomaly"]].copy()
anom_pdf["причина"] = np.where(anom_pdf.temp_z.abs() >= 3, "температура (|z| ≥ 3)", "вибрация (RMS ≥ 0,45)")
fig2 = px.scatter(anom_pdf, x="timestamp", y="temperature", color="причина", symbol="robot_id",
                  title="События, классифицированные как аномальные (цвет — сработавшее правило)")
fig2.update_layout(template="simple_white", height=380, xaxis_title="время", yaxis_title="температура в момент тревоги, °C")
fig2.show()

# Крупный план: Z-score и RMS одного робота около аномалий
r0 = plot_df[plot_df.robot_id == "robot-0"].reset_index(drop=True)
fig3 = make_subplots(rows=1, cols=2, subplot_titles=("robot-0: z температуры, 830–890 с", "robot-0: RMS вибрации, 1180–1260 с"))
a = r0.iloc[830:890]; b = r0.iloc[1180:1260]
fig3.add_trace(go.Scatter(x=a.timestamp, y=a.temp_z, mode="lines+markers", marker=dict(size=4), name="z", line=dict(color="#2a78d6")), 1, 1)
fig3.add_hline(y=3, line_dash="dash", line_color="#e34948", line_width=2, opacity=1, row=1, col=1)
fig3.add_trace(go.Scatter(x=b.timestamp, y=b.vibration_rms, mode="lines+markers", marker=dict(size=4), name="RMS", line=dict(color="#eb6834")), 1, 2)
fig3.add_hline(y=0.45, line_dash="dash", line_color="#e34948", line_width=2, opacity=1, row=1, col=2)
for col, (s0, s1) in ((1, (850, 870)), (2, (1200, 1225))):
    fig3.add_vrect(x0=start + pd.Timedelta(seconds=s0), x1=start + pd.Timedelta(seconds=s1),
                   fillcolor="orange", opacity=0.25, line_width=0, row=1, col=col)
fig3.update_layout(template="simple_white", height=360, showlegend=False)
fig3.show()

**Как читать результат.**

* **График 1 (RMS вибрации):** красный пунктир — порог 0,45, оранжевая полоса — настоящая аномалия. Смотрите, где линия RMS пересекает порог относительно полосы.
* **График 2 (точки тревог):** каждая точка — момент тревоги на шкале температуры; цвет — какое правило сработало. Точки «по вибрации» стоят на обычной температуре — это не ошибка, тревога вызвана вибрацией.
* **График 3 (крупный план):** слева z — «прямоугольник», потому что каждая секунда перегрева сразу даёт большой z; справа RMS — «треугольник», потому что рамка наполняется и опустошается постепенно.

**Что получено.** Допишите:

> На первом графике RMS пересекает порог через 4 с после начала аномалии и опускается ниже порога через 16 с после её конца.
> На втором графике тревог по температуре 60, по вибрации 111. Часть точек стоит на нормальной температуре, потому что в этих моментах сработало правило по вибрации, а не по температуре.
> На крупном плане аномалия по z выглядит как прямоугольник, а по RMS — как треугольник, потому что z реагирует сразу на каждый аномальный отсчёт, а RMS меняется постепенно: окно из 20 точек сначала заполняется аномальными значениями, а потом постепенно очищается.

---
# Шаг 7. Чувствительность порогов и компромисс FP/FN (2 балла вместе с графиками)

## Ячейка 7. Сетка порогов, точность по отсчётам и по событиям

**Что делает ячейка.** Для каждого правила **отдельно** перебирает пороги и считает TP, FP, FN, precision, recall и $F_1$ относительно разметки истины. Сравнивает три способа калибровки Z-score и два вида окна RMS, а затем считает метрики на уровне событий.

**Задание.** Допишите **5 строк**.

* **TODO 7.1** — три строки подсчёта TP, FP, FN в функции `confusion`. Образец — таблица «по причинам» в ячейке 5: `F.sum(условие.cast("int"))` считает, сколько строк удовлетворяют условию (True = 1, False = 0).
* **TODO 7.2** — два списка порогов, **не меньше 5 значений** в каждом. Можно взять пример из подсказки или свои.

**Что такое precision и recall простыми словами:** *precision* — какая доля тревог была настоящей (мало ложных тревог); *recall* — какую долю настоящих аномалий нашли (мало пропусков). Ячейка работает 1–2 минуты.

In [ ]:
# 7. Чувствительность порогов: точность по каждому правилу отдельно, по разметке истины.
def confusion(flag_col, truth_col):
    tp_expr = F.sum((flag_col & truth_col).cast("int")).alias("TP")   # TODO 7.1 ← впишите: F.sum((flag_col & truth_col).cast("int")).alias("TP")     (тревога и истина)
    fp_expr =F.sum((flag_col & ~truth_col).cast("int")).alias("FP")   #          ← впишите: F.sum((flag_col & ~truth_col).cast("int")).alias("FP")    (тревога без истины)
    fn_expr = F.sum((~flag_col & truth_col).cast("int")).alias("FN")   #          ← впишите: F.sum((~flag_col & truth_col).cast("int")).alias("FN")    (истина без тревоги)
    check_filled("ячейка 7, TODO 7.1", {"tp_expr": tp_expr, "fp_expr": fp_expr, "fn_expr": fn_expr})
    r = sdf.agg(tp_expr, fp_expr, fn_expr).first()
    tp, fp, fn = r["TP"], r["FP"], r["FN"]
    p = tp / (tp + fp) if tp + fp else float("nan"); rc = tp / (tp + fn) if tp + fn else float("nan")
    return dict(TP=tp, FP=fp, FN=fn, precision=p, recall=rc, F1=2 * p * rc / (p + rc) if p and rc else 0.0)

Z_THRESHOLDS = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0]     # TODO 7.2 ← впишите, например: [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0]
RMS_THRESHOLDS = [0.25, 0.30, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85]   #          ← впишите, например: [0.25, 0.30, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85]
check_filled("ячейка 7, TODO 7.2", {"Z_THRESHOLDS": Z_THRESHOLDS, "RMS_THRESHOLDS": RMS_THRESHOLDS})
assert len(Z_THRESHOLDS) >= 5 and len(RMS_THRESHOLDS) >= 5, "TODO 7.2: в каждом списке порогов нужно не меньше 5 значений"

z_rows = [dict(порог=z, **confusion(F.abs("temp_z") >= z, F.col("true_temp"))) for z in Z_THRESHOLDS]
zb_rows = [dict(порог=z, **confusion(F.abs("temp_z_base") >= z, F.col("true_temp"))) for z in [3.0, 5.0, 8.0, 12.0]]
rms_rows = [dict(порог=t, **confusion(F.col("vibration_rms") >= t, F.col("true_vib"))) for t in RMS_THRESHOLDS]
zr_rows = [dict(порог=z, **confusion(F.abs("temp_z_roll") >= z, F.col("true_temp"))) for z in [3.0, 4.0, 5.0, 8.0]]
rmsc_rows = [dict(порог=t, **confusion(F.col("vibration_rms_c") >= t, F.col("true_vib"))) for t in [0.45, 0.55, 0.65]]
z_tab, zb_tab, rms_tab = pd.DataFrame(z_rows), pd.DataFrame(zb_rows), pd.DataFrame(rms_rows)
zr_tab, rmsc_tab = pd.DataFrame(zr_rows), pd.DataFrame(rmsc_rows)
print("Z-score температуры (μ, σ по всему ряду, как в задании):"); print(z_tab.round(3).to_string(index=False))
print("\nZ-score, откалиброванный по первым 10 минутам:"); print(zb_tab.round(3).to_string(index=False))
print("\nZ-score по скользящему окну 300 с (локальная калибровка):"); print(zr_tab.round(3).to_string(index=False))
print("\nRolling RMS вибрации (окно задания: 19 с назад + текущий отсчёт):"); print(rms_tab.round(3).to_string(index=False))
print("\nRMS по центрированному окну (офлайн):"); print(rmsc_tab.round(3).to_string(index=False))

# Уровень событий: связные интервалы тревоги; событие истинное, если пересекается с интервалом аномалии
def events(flag, truth):
    pdx = sdf.select("robot_id", F.col("timestamp").cast("long").alias("ts"), flag.alias("f"), truth.alias("t")).toPandas()
    rows = []
    for robot, g in pdx.sort_values("ts").groupby("robot_id"):
        seg = (g.f != g.f.shift()).cumsum()
        for _, s in g[g.f].groupby(seg[g.f]):
            rows.append(dict(robot_id=robot, start_s=int(s.ts.min() - start.timestamp()),
                             end_s=int(s.ts.max() - start.timestamp()), true_event=bool(s.t.any())))
    return pd.DataFrame(rows)
ev_t = events(F.abs("temp_z") >= 3.0, F.col("true_temp")); ev_v = events(F.col("vibration_rms") >= 0.45, F.col("true_vib"))
print(f"\nСобытия (порог задания): температура — {len(ev_t)} интервалов, из них истинных {ev_t.true_event.sum()}; "
      f"вибрация — {len(ev_v)} интервалов, из них истинных {ev_v.true_event.sum()}")
print(ev_v.to_string(index=False))

fig4 = make_subplots(rows=1, cols=2, subplot_titles=("Температура: порог |z|", "Вибрация: порог RMS"))
for col, tab in ((1, z_tab), (2, rms_tab)):
    for metric, color in (("precision", "#2a78d6"), ("recall", "#eb6834"), ("F1", "#1baf7a")):
        fig4.add_trace(go.Scatter(x=tab["порог"], y=tab[metric], mode="lines+markers", name=metric,
                                  line=dict(color=color), showlegend=(col == 1)), 1, col)
fig4.add_vline(x=3.0, line_dash="dash", line_color="#999", line_width=2, opacity=1, row=1, col=1)
fig4.add_vline(x=0.45, line_dash="dash", line_color="#999", line_width=2, opacity=1, row=1, col=2)
fig4.update_yaxes(range=[0, 1.05])
fig4.update_layout(template="simple_white", height=380, title="Компромисс FP/FN: пунктир — пороги задания")
fig4.show()

Z-score температуры (μ, σ по всему ряду, как в задании):
 порог  TP  FP  FN  precision  recall    F1
   1.5  60   8   0      0.882     1.0 0.938
   2.0  60   0   0      1.000     1.0 1.000
   2.5  60   0   0      1.000     1.0 1.000
   3.0  60   0   0      1.000     1.0 1.000
   3.5  60   0   0      1.000     1.0 1.000
   4.0  60   0   0      1.000     1.0 1.000
   5.0  60   0   0      1.000     1.0 1.000
   6.0  60   0   0      1.000     1.0 1.000

Z-score, откалиброванный по первым 10 минутам:
 порог  TP   FP  FN  precision  recall    F1
   3.0  60 2635   0      0.022     1.0 0.044
   5.0  60 1375   0      0.042     1.0 0.080
   8.0  60   18   0      0.769     1.0 0.870
  12.0  60    0   0      1.000     1.0 1.000

Z-score по скользящему окну 300 с (локальная калибровка):
 порог  TP  FP  FN  precision  recall    F1
   3.0  60  37   0      0.619   1.000 0.764
   4.0  54   1   6      0.982   0.900 0.939
   5.0  34   0  26      1.000   0.567 0.723
   8.0  16   0  44      1.000   0.267 0

**Как читать результат.**

* В каждой таблице одна строка — один порог. **Низкий порог** — правило «нервное»: находит всё (recall высокий), но часто ошибается (precision низкий). **Высокий порог** — правило «спокойное»: мало ложных тревог, но пропускает аномалии.
* Сравните три таблицы Z-score: калибровка «по всему ряду» (задание), «по первым 10 минутам» и «по скользящему окну 300 с».
* Таблица RMS: даже при высоком пороге precision не становится 1 — мешает «хвост» скользящего окна. Центрированное окно (офлайн) хвоста почти не имеет.
* «События» — то же, но считаем не секунды, а целые эпизоды тревоги: событие засчитано, если оно пересекается с настоящей аномалией.
* График: синяя линия — precision, оранжевая — recall, зелёная — F1 (баланс между ними); пунктир — порог задания.

**Что получено.** Допишите своими числами:

> Для Z-score при пороге 1,5 recall = 1,0, precision = 0,882; при пороге 3,0 — recall = 1,0, precision = 1,0. Лучший F1 = 1,0 при пороге  3,0.
> Для RMS лучший F1 = 0,725 при пороге 0,25; precision не поднимается выше 0,750, потому что скользящее окно продолжает учитывать повышенные значения вибрации после окончания аномалии, вызывая ложные срабатывания.
> По событиям правило задания нашло 3 из 3 аномалий по вибрации и дало 0 ложных событий — по событиям качество выглядит лучше, чем по секундам, потому что задержка срабатывания и последующий «хвост» не мешают распознать весь эпизод как одно истинное событие.
> Для робота я бы выбрала порог 0,25 для RMS, потому что важнее не пропустить аномалию.

## Ячейка 8. Сводка

**Что делает ячейка.** Выполняет сводку задания (доля аномалий по роботам), считает качество итогового правила и выполняет `assert` задания.

**Задание.** Ячейка готова — просто запустите.

In [ ]:
# 6. Сводка (код задания) + качество итогового правила.
summary = (
    sdf.groupBy("robot_id")
    .agg(
        F.count("*").alias("total"),
        F.sum(F.col("is_anomaly").cast("int")).alias("anomalies")
    )
    .withColumn("anomaly_rate", F.col("anomalies") / F.col("total"))
)
summary.orderBy("robot_id").show()

overall = confusion(F.col("is_anomaly"), F.col("true_temp") | F.col("true_vib"))
print("Итоговое правило задания относительно истины:", {k: round(v, 3) for k, v in overall.items()})
assert anomalies.count() > 0

+--------+-----+---------+-------------------+
|robot_id|total|anomalies|       anomaly_rate|
+--------+-----+---------+-------------------+
| robot-0| 1800|       57|0.03166666666666667|
| robot-1| 1800|       57|0.03166666666666667|
| robot-2| 1800|       57|0.03166666666666667|
+--------+-----+---------+-------------------+

Итоговое правило задания относительно истины: {'TP': 123, 'FP': 48, 'FN': 12, 'precision': 0.719, 'recall': 0.911, 'F1': 0.804}


**Как читать результат.** `anomaly_rate` — доля секунд, отмеченных как аномальные; строка «Итоговое правило» — качество правила задания (обе причины вместе) относительно «правильных ответов».

**Что получено.** Допишите своими числами:

> Доля аномалий у каждого робота =0,0317 (≈ 3,17 % времени). Итоговое правило: precision = 0,719, recall =0.911,, F1 = 0.804.
> Это значит, что из 10 тревог примерно 7 настоящие, а из 10 настоящих аномальных секунд правило находит 9.

## Ячейка 9. Самопроверка

**Что делает ячейка.** Проверяет наличие обязательных результатов. Это «предполётный контроль», а не оценка.

**Задание.** Ячейка готова — просто запустите. Все пункты должны быть `OK`; если где-то `НЕТ`, вернитесь к названной ячейке.

In [ ]:
checks = {
    "Заполнены ФИО и группа":                   STUDENT_NAME != "Иванов Иван Иванович",
    "Телеметрия: 3 робота × 1800 отсчётов":     len(pdf) == 3 * 1800,
    "Z-score по каждому роботу (partitionBy)":  stats["temp_std"].nunique() == 3,
    "σ совпадает с pandas (stddev_pop)":        np.allclose(stats["temp_std"], stats["pandas_std"]),
    "Rolling RMS по окну 20 строк":             int(sdf.agg(F.max("rms_window_n")).first()[0]) == 20,
    "Таблица аномалий получена":                anomalies.count() > 0,
    "Анализ чувствительности: ≥ 5 порогов":     len(z_tab) >= 5 and len(rms_tab) >= 5,
    "Метрики precision/recall посчитаны":       {"precision", "recall"} <= set(z_tab.columns),
}
for name, ok in checks.items():
    print(f"{'OK ' if ok else 'НЕТ'}  {name}")
print(f"\nВыполнено: {sum(checks.values())} из {len(checks)}")

OK   Заполнены ФИО и группа
OK   Телеметрия: 3 робота × 1800 отсчётов
OK   Z-score по каждому роботу (partitionBy)
OK   σ совпадает с pandas (stddev_pop)
OK   Rolling RMS по окну 20 строк
OK   Таблица аномалий получена
OK   Анализ чувствительности: ≥ 5 порогов
OK   Метрики precision/recall посчитаны

Выполнено: 8 из 8


**Что получено.** Допишите:

> Самопроверка: выполнено 8 из 8.

---

# TEACH CARD (2 балла)

Заполните паспорт педагогической адаптации выполненной инженерной задачи. Формулируйте для школьника **без терминов «Spark», «Z-score», «окно»** и опирайтесь на числа своего прогона. В третьем столбце — подсказка, о чём писать; заполните **второй** столбец.

| Поле | Содержание | Подсказка |
|---|---|---|
| **Название учебного проекта** | «Робот-сторож: как вовремя заметить неисправность?» | короткое и понятное школьнику, например «Робот-сторож: заметь, когда мотор перегрелся» |
| **Целевая аудитория** (8–9 класс / 10–11 класс / кружок робототехники / СПО) |  8–9 класс | выберите одну и учтите её математическую подготовку (среднее и разброс знают с 7–8 класса) |
| **Исследовательский вопрос** | При какой границе срабатывания робот обнаруживает не менее 90% аномальных измерений и допускает как можно меньше ложных тревог?| вопрос с ответом-числом: «при каком пороге робот-сторож почти не пропускает перегрев и редко поднимает ложную тревогу?» |
| **Источник данных** | Искусственно созданные показания температуры и вибрации трёх роботов за 30 минут (5400 измерений) | датчик температуры и акселерометр учебного робота (Arduino, LEGO, micro:bit) или набор из этой работы |
| **Инженерная концепция, сохраняемая без упрощения** |Чем чувствительнее система, тем меньше неисправностей она пропускает, но тем чаще возникают ложные тревоги. Поэтому необходимо сначала определить нормальные показания робота, а затем подобрать границу срабатывания. | одна «настоящая» мысль: порог — это компромисс между пропусками и ложными тревогами; норму нужно измерять на нормальной работе |
| **Что упрощается относительно университетской лабораторной** |Исследуется один робот вместо трёх. Данные обрабатываются в обычной таблице. Школьники сравнивают показания со средним значением и анализируют изменение вибрации за последние 20 секунд  | один робот, таблица вместо Spark, «отклонение от среднего в разах разброса» вместо z, среднее за 20 секунд вместо RMS |
| **Алгоритм действий обучающегося (4–6 шагов)** | 1. Получить показания датчиков робота. 2. Найти среднюю температуру и обычный разброс значений. 3. Выбрать границу срабатывания тревоги. 4. Отметить обнаруженные неисправности. 5. Сравнить результаты с реальными неисправностями. 6. Подобрать наиболее подходящую границу срабатывания | записать показания → найти обычное значение и разброс → задать порог → отметить тревоги → сравнить с тем, что было на самом деле → выбрать лучший порог |
| **Измеримый результат / метрика** | Доля правильно обнаруженных неисправностей, количество ложных тревог и задержка срабатывания. В лабораторной обнаружено 91,1% аномальных отсчётов, 71,9% тревог оказались правильными | доля найденных аномалий и доля ложных тревог (понятные аналоги recall и precision), задержка тревоги в секундах |
| **Критерий успешного выполнения** | Обнаружено не менее 90% аномальных измерений, задержка срабатывания не превышает 5 секунд, не менее 70% тревог являются правильными. Ученик может объяснить причины ложных срабатываний. | конкретное число: «находит перегрев не позже чем через 5 с, ложных тревог не больше 1 за 10 минут» |
| **Вариант усложнения для НТО / хакатона** | Подключить настоящие датчики температуры и вибрации к учебному роботу и настроить световой или звуковой сигнал при обнаружении неисправности | реальные датчики, тревога в реальном времени (светодиод, звук, сообщение), несколько датчиков сразу |
| **Риски и ограничения** |Температура двигателя постепенно растёт даже при нормальной работе. Из-за этого могут возникать ложные тревоги. Кроме того, настройки одного робота не всегда подходят другому  | мотор нагревается и в норме — постоянный порог «устаревает»; ложные тревоги приучают их игнорировать; данные одного робота не переносятся на другой |